In [0]:
from pyspark.sql import functions as F


SILVER_PATH = "/Volumes/ecommerce_catalog/ecommerce_schema/silver"

CUSTOMERS_PATH = f"{SILVER_PATH}/clean_customers/clean_customers.csv"
ORDERS_PATH = f"{SILVER_PATH}/clean_orders/clean_orders.csv"

customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CUSTOMERS_PATH)
)


orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ORDERS_PATH)
)


orders = orders.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unit_price")
)

# completed orders
completed_orders = orders.filter(
    F.col("status") == "completed"
)

print(f"Completed orders: {completed_orders.count()}")

completed_count = completed_orders.count()

print(f"Completed orders available for reporting: {completed_count}")


# trusted total revenue 
total_revenue = (
    completed_orders
    .agg(
        F.sum("total_amount").alias("trusted_total_revenue")
    )
)

display(total_revenue)


# revenue by city 

revenue_by_city = (
    completed_orders
    .groupBy("city")
    .agg(
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

display(revenue_by_city)

# revenue by product category

revenue_by_category = (
    completed_orders
    .groupBy("product_category")
    .agg(
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

display(revenue_by_category)

# top 10 orders

top_10_orders = (
    completed_orders
    .select(
        "order_id",
        "customer_id",
        "city",
        "product_category",
        "quantity",
        "unit_price",
        "total_amount"
    )
    .orderBy(F.desc("total_amount"))
    .limit(10)
)

display(top_10_orders)

# avg completed order value

average_order_value = (
    completed_orders
    .agg(
        F.avg("total_amount").alias("average_order_value")
    )
)

display(average_order_value)


# orders by status

orders_by_status = (
    orders
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

display(orders_by_status)


# top cities by revenuee
top_5_cities = (
    completed_orders
    .groupBy("city")
    .agg(
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
    .limit(5)
)

display(top_5_cities)

**Customer** **Enrichment**

In [0]:
from pyspark.sql import functions as F


SILVER_PATH = "/Volumes/ecommerce_catalog/ecommerce_schema/silver"

CUSTOMERS_PATH = f"{SILVER_PATH}/clean_customers/clean_customers.csv"
ORDERS_PATH = f"{SILVER_PATH}/clean_orders/clean_orders.csv"

customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(CUSTOMERS_PATH)
)


orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ORDERS_PATH)
)


orders = orders.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unit_price")
)


# revenue by costumer

customer_orders = (
    completed_orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("c.customer_name"),
        F.col("c.customer_type"),
        F.col("o.city"),
        F.col("o.product_category"),
        F.col("o.total_amount")
    )
)
# top costumers
display(customer_orders.limit(10))


# revenue by customer type

revenue_by_customer_type = (
    customer_orders
    .groupBy("customer_type")
    .agg(
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

display(revenue_by_customer_type)

# Orders that cannot be associated with a valid customer

orders_without_customer = (
    completed_orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
)

print(
    f"Orders without valid customer: "
    f"{orders_without_customer.count()}"
)

display(orders_without_customer)


# order value classification 

orders_classified = completed_orders.withColumn(
    "order_size",
    F.when(F.col("total_amount") < 50, "small")
     .when(F.col("total_amount") < 200, "medium")
     .when(F.col("total_amount") < 500, "large")
     .otherwise("premium")
)

order_size_counts = (
    orders_classified
    .groupBy("order_size")
    .count()
    .orderBy(
        F.when(F.col("order_size") == "small", 1)
         .when(F.col("order_size") == "medium", 2)
         .when(F.col("order_size") == "large", 3)
         .when(F.col("order_size") == "premium", 4)
    )
)

display(order_size_counts)

